# [7.4] Mini Natural Language Autoencoders - Exercises

Natural Language Autoencoders try to compress activations into short text and then reconstruct activations from that text. In this local section, the "text" is deliberately constrained: a phrase-bank bottleneck, not a free-form language model. That makes the method small enough to inspect.

You will build the evaluation loop used by the accepted CUDA report:

```text
activation -> phrase id -> reconstructed activation -> behavior / latent / control checks
```

The section is course-ready only if you can explain why each control matters, not merely if the tests print green.


In [ ]:
import json
import sys
from dataclasses import dataclass
from pathlib import Path

import matplotlib.pyplot as plt
import torch as t
import torch.nn.functional as F

chapter = "chapter7_activation_to_language"
section = "part4_mini_natural_language_autoencoders"
root_dir = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / chapter).exists())
exercises_dir = root_dir / chapter / "exercises"
section_dir = exercises_dir / section

if str(root_dir) not in sys.path:
    sys.path.append(str(root_dir))
if str(exercises_dir) not in sys.path:
    sys.path.append(str(exercises_dir))

import part4_mini_natural_language_autoencoders.tests as tests

GT_TIER = "GT-3"
EXERCISE_ID = "7_4_mini_natural_language_autoencoders"
EXPECTED_RUNTIME = "25-45 minutes for exercises; about 1 minute for the CUDA preflight"
REQUIRES_GPU = True


## Text-Bottleneck Records

The first task is bookkeeping, but it is consequential. Every activation row must stay aligned with three kinds of text: the original prompt span, the latent label, and the generated phrase that crosses the bottleneck.

> ```yaml
> Difficulty: 🟢⚪⚪⚪⚪
> Importance: 🔵🔵🔵🔵⚪
> You should spend 5-10 minutes on this exercise.
> ```

<details>
<summary>Expected output</summary>

```text
All tests in `test_build_nla_training_batch_validates_alignment` passed!
All tests in `test_build_nla_training_batch_rejects_empty_batches` passed!
```

The valid fixture has activation shape `[3, 3]` and phrases like `indirect object is Bob`.

</details>

<details>
<summary>Help - what is the bottleneck text?</summary>

The original prompt is not the bottleneck. The generated phrase is. If those two fields get mixed up, prompt copying can masquerade as activation explanation.

</details>

<details>
<summary>Common bug</summary>

A rank-1 tensor like `[d_model]` is not a batch. Use `[examples, d_model]`, even if there is only one example.

</details>

<details>
<summary>Solution</summary>

Validate rank and nonempty batch size, check all text lists have one row per activation, and return immutable tuples so later code cannot silently reorder rows.

</details>


In [ ]:
@dataclass(frozen=True)
class NLATrainingBatch:
    activations: t.Tensor
    original_text_spans: tuple[str, ...]
    synthetic_latent_labels: tuple[str, ...]
    generated_explanations: tuple[str, ...]


def build_nla_training_batch(
    activations: t.Tensor,
    original_text_spans: list[str],
    synthetic_latent_labels: list[str],
    generated_explanations: list[str],
) -> NLATrainingBatch:
    """Bundle activations with source spans, labels, and generated explanations."""
    raise NotImplementedError()


tests.test_build_nla_training_batch_validates_alignment(build_nla_training_batch)
tests.test_build_nla_training_batch_rejects_empty_batches(build_nla_training_batch)


## Reconstruction Quality

A phrase bottleneck should reconstruct activations better than a baseline that never saw the activation. This is the first quantitative gate.

> ```yaml
> Difficulty: 🟡🟡⚪⚪⚪
> Importance: 🔵🔵🔵🔵⚪
> You should spend 10-15 minutes on this exercise.
> ```

<details>
<summary>Expected output</summary>

```text
All tests in `test_activation_reconstruction_report_beats_text_only_baseline` passed!
All tests in `test_activation_reconstruction_report_rejects_empty_and_rank1_inputs` passed!
```

The toy fixture should report `activation_mse ~= 0.01`, `text_only_mse = 0.5`, and cosine similarity above `0.99`.

</details>

<details>
<summary>Help - why MSE is only the first gate</summary>

MSE notices whether the vector is close on average. It does not tell you whether a behaviorally important direction moved. That is why the next exercise checks a target logit difference.

</details>

<details>
<summary>Common bug</summary>

Do not let PyTorch broadcasting rescue mismatched shapes. All three tensors must have exactly the same shape.

</details>

<details>
<summary>Solution</summary>

Use `F.mse_loss` for the reconstruction and baseline, flatten per example for cosine similarity, and reject empty or rank-1 activation tensors.

</details>


In [ ]:
@dataclass(frozen=True)
class NLAReconstructionReport:
    activation_mse: float
    text_only_mse: float
    mean_cosine_similarity: float
    beats_text_only: bool


def activation_reconstruction_report(
    original_activations: t.Tensor,
    reconstructed_activations: t.Tensor,
    text_only_reconstructions: t.Tensor,
) -> NLAReconstructionReport:
    raise NotImplementedError()


tests.test_activation_reconstruction_report_beats_text_only_baseline(
    activation_reconstruction_report,
)
tests.test_activation_reconstruction_report_rejects_empty_and_rank1_inputs(
    activation_reconstruction_report,
)


## Behavioral Preservation

A low reconstruction error can still break the computation you care about. Here you check one target behavior: a positive-minus-negative logit difference.

> ```yaml
> Difficulty: 🟡🟡⚪⚪⚪
> Importance: 🔵🔵🔵🔵🔵
> You should spend 10 minutes on this exercise.
> ```

<details>
<summary>Expected output</summary>

```text
All tests in `test_logit_diff_preservation_report_checks_actual_logit_diff` passed!
All tests in `test_logit_diff_preservation_report_rejects_bad_inputs` passed!
```

The toy mean absolute logit-diff error should be close to `0.15`.

</details>

<details>
<summary>Help - interpreting the logit-diff check</summary>

This is stronger than MSE but narrower than full behavior. It says the reconstruction preserved one selected readout, not that all downstream completions would match.

</details>

<details>
<summary>Common bug</summary>

Compute absolute error per example and then average. Comparing only batch means can hide paired failures.

</details>

<details>
<summary>Solution</summary>

Validate batched logits and non-negative tolerances, compute `positive - negative` for each example, then average the absolute differences.

</details>


In [ ]:
@dataclass(frozen=True)
class LogitDiffPreservationReport:
    original_logit_diff: float
    reconstructed_logit_diff: float
    mean_abs_error: float
    preserves_target_logit_diff: bool


def batch_target_logit_diff(
    logits: t.Tensor,
    *,
    positive_token_id: int,
    negative_token_id: int,
) -> t.Tensor:
    raise NotImplementedError()


def logit_diff_preservation_report(
    original_logits: t.Tensor,
    reconstructed_logits: t.Tensor,
    *,
    positive_token_id: int,
    negative_token_id: int,
    max_mean_abs_error: float = 0.25,
) -> LogitDiffPreservationReport:
    raise NotImplementedError()


tests.test_logit_diff_preservation_report_checks_actual_logit_diff(
    logit_diff_preservation_report,
)
tests.test_logit_diff_preservation_report_rejects_bad_inputs(
    logit_diff_preservation_report,
)


## Latent Preservation

Now check whether a probe-decoded latent survives the bottleneck. You need both accuracy against labels and agreement with the original activation readout.

> ```yaml
> Difficulty: 🟡🟡⚪⚪⚪
> Importance: 🔵🔵🔵🔵🔵
> You should spend 10 minutes on this exercise.
> ```

<details>
<summary>Expected output</summary>

```text
All tests in `test_latent_preservation_report_requires_accuracy_and_agreement` passed!
All tests in `test_latent_preservation_report_rejects_invalid_thresholds` passed!
```

The positive toy case has original accuracy `1.0`, reconstructed accuracy `1.0`, and agreement `1.0`.

</details>

<details>
<summary>Help - why accuracy and agreement both matter</summary>

Agreement alone can preserve a wrong answer. Accuracy alone can hide that the reconstruction changed the internal readout on individual examples. You want both.

</details>

<details>
<summary>Common bug</summary>

Check `logits.shape[:-1] == labels.shape`. This keeps batched and higher-rank probe outputs honest.

</details>

<details>
<summary>Solution</summary>

Use top-1 predictions for both original and reconstructed probe logits, validate thresholds are probabilities, then require reconstructed accuracy and prediction agreement to clear their thresholds.

</details>


In [ ]:
@dataclass(frozen=True)
class LatentPreservationReport:
    original_probe_accuracy: float
    reconstructed_probe_accuracy: float
    prediction_agreement: float
    preserves_latents: bool


def _prediction_accuracy(logits: t.Tensor, labels: t.Tensor) -> float:
    raise NotImplementedError()


def latent_preservation_report(
    original_probe_logits: t.Tensor,
    reconstructed_probe_logits: t.Tensor,
    latent_ids: t.Tensor,
    *,
    min_accuracy: float = 0.75,
    min_agreement: float = 0.75,
) -> LatentPreservationReport:
    raise NotImplementedError()


tests.test_latent_preservation_report_requires_accuracy_and_agreement(
    latent_preservation_report,
)
tests.test_latent_preservation_report_rejects_invalid_thresholds(
    latent_preservation_report,
)


## Compression And Counterfactuals

Two shortcut controls are cheap and useful. The phrase should be shorter than the prompt, and a large counterfactual activation change should change the phrase.

> ```yaml
> Difficulty: 🟡🟡⚪⚪⚪
> Importance: 🔵🔵🔵🔵⚪
> You should spend 10-15 minutes on this exercise.
> ```

<details>
<summary>Expected output</summary>

```text
All tests in `test_brevity_and_counterfactual_reports_reject_prompt_copying` passed!
All tests in `test_brevity_and_counterfactual_reports_reject_bad_controls` passed!
```

The toy explanation pair should have compression ratio below `0.5`, and the Bob-to-Alice counterfactual should change.

</details>

<details>
<summary>Help - what this does not prove</summary>

Short text can be nonsense, and changed text can be arbitrary. These checks only rule out two easy failures. The full signature result also checks numeric payloads, shuffled phrases, and blank phrases.

</details>

<details>
<summary>Common bug</summary>

Do not count whitespace or case-only edits as explanation changes. Normalize strings before comparing.

</details>

<details>
<summary>Solution</summary>

Count words with simple whitespace splitting, reject empty explanations, require generated word count to be strictly lower, and require activation delta to exceed the requested threshold.

</details>


In [ ]:
@dataclass(frozen=True)
class GeneratedTextBrevityReport:
    generated_word_count: int
    original_word_count: int
    compression_ratio: float
    shorter_than_original: bool


@dataclass(frozen=True)
class CounterfactualExplanationReport:
    original_explanation: str
    counterfactual_explanation: str
    activation_delta: float
    explanation_changed: bool


def generated_text_brevity_report(
    generated_explanations: list[str],
    original_prompts: list[str],
) -> GeneratedTextBrevityReport:
    raise NotImplementedError()


def counterfactual_explanation_report(
    original_activation: t.Tensor,
    counterfactual_activation: t.Tensor,
    original_explanation: str,
    counterfactual_explanation: str,
    *,
    min_activation_delta: float = 0.0,
) -> CounterfactualExplanationReport:
    raise NotImplementedError()


tests.test_brevity_and_counterfactual_reports_reject_prompt_copying(
    generated_text_brevity_report,
    counterfactual_explanation_report,
)
tests.test_brevity_and_counterfactual_reports_reject_bad_controls(
    generated_text_brevity_report,
    counterfactual_explanation_report,
)


## Trainable Discrete Bottleneck

This is the miniature NLA. Train an activation-to-phrase classifier, train a phrase-id-to-activation decoder table, and evaluate on held-out activations.

> ```yaml
> Difficulty: 🔴🔴🔴⚪⚪
> Importance: 🔵🔵🔵🔵🔵
> You should spend 20-25 minutes on this exercise.
> ```

<details>
<summary>Expected output</summary>

```text
All tests in `test_trainable_discrete_bottleneck_learns_phrase_ids` passed!
All tests in `test_trainable_discrete_bottleneck_rejects_empty_splits` passed!
```

The toy bottleneck should learn both phrase ids, produce `eval_phrase_accuracy = 1.0`, and beat a blank-text mean reconstruction.

</details>

<details>
<summary>Help - model sketch</summary>

Use a linear classifier from centered normalized activations to phrase ids. Use a decoder table initialized from phrase-wise train activation means. Optimize cross-entropy plus reconstruction MSE.

</details>

<details>
<summary>Common bug</summary>

Do not decode from the true eval phrase id. Decode from the predicted phrase id. Otherwise the evaluation leaks the answer.

</details>

<details>
<summary>Solution</summary>

Follow the reference solution in `solutions.py`: validate shapes and ids, seed Torch, train `encoder_weight`, `encoder_bias`, and `decoder_table`, then return detached weights, eval predictions, reconstructed eval activations, and a report dataclass.

</details>


In [ ]:
@dataclass(frozen=True)
class TrainableNLABottleneckReport:
    encoder_final_loss: float
    decoder_final_mse: float
    encoder_train_accuracy: float
    eval_phrase_accuracy: float
    reconstruction_mse: float
    blank_text_mse: float
    beats_blank_text: bool
    generated_explanations: tuple[str, ...]
    phrase_count: int
    training_steps: int
    seed: int


def train_discrete_nla_bottleneck(
    train_activations: t.Tensor,
    train_phrase_ids: t.Tensor,
    eval_activations: t.Tensor,
    eval_phrase_ids: t.Tensor,
    phrase_texts: tuple[str, ...],
    *,
    steps: int = 300,
    lr: float = 0.05,
    seed: int = 0,
) -> tuple[t.Tensor, t.Tensor, t.Tensor, t.Tensor, t.Tensor, TrainableNLABottleneckReport]:
    raise NotImplementedError()


tests.test_trainable_discrete_bottleneck_learns_phrase_ids(
    train_discrete_nla_bottleneck,
)
tests.test_trainable_discrete_bottleneck_rejects_empty_splits(
    train_discrete_nla_bottleneck,
)


## Combined Contract

Compose the helpers into one CPU-only report. This is the notebook-level contract: if an earlier helper drifts, the combined report should fail loudly.

<details>
<summary>Expected output</summary>

```text
All tests in `test_notebook_contract` passed!
```

The combined report should include reconstruction, logit diff, latent preservation, brevity, counterfactual, and trainable bottleneck entries.

</details>

<details>
<summary>Help - why this is useful</summary>

The combined report is a miniature version of the release report. It makes the claim auditable as a sequence rather than a pile of isolated helper tests.

</details>

<details>
<summary>Common bug</summary>

It is easy to forget to include the trainable bottleneck result. The final report should not stop at metric helper functions.

</details>

<details>
<summary>Solution</summary>

Use the same toy fixtures from the individual tests, call each report helper once, and return a nested dictionary with stable keys.

</details>


In [ ]:
def run_smoke_test(cpu: bool = True) -> dict:
    _ = cpu
    raise NotImplementedError()


tests.test_notebook_contract(run_smoke_test)


## Signature Result

The final classroom result is the committed CUDA report from a pinned `gelu-1l` run. This cell is a fast report inspection path. For live CUDA regeneration, use `run_full_experiment()` below.

<details>
<summary>Expected output</summary>

A reconstruction bar chart should show NLA MSE below text-only and prompt-label MSE, with shuffled text much worse. The summary table should include `numeric_literal_count = 0`, NLA latent accuracy `1.0`, text-only accuracy `0.5`, and peak VRAM below `1 GB`.

</details>

<details>
<summary>Help - report replay vs live CUDA</summary>

The committed report keeps the notebook fast and CPU-viewable. It is not a substitute for the release gate. `run_full_experiment()` calls the live CUDA implementation from `solutions.py`.

</details>

<details>
<summary>Common bug</summary>

Do not treat the report as evidence for a full NLA. It is evidence for a 12-phrase discrete phrase-bank preflight on one `gelu-1l` hook.

</details>


In [ ]:
def _load_committed_gpu_report() -> dict:
    report = json.loads((section_dir / "verification_report.json").read_text())
    assert report["accepted"] and report["tests_passed"]
    gpu = report["metrics"]["gpu_test"]
    required = [
        "preflight_passed",
        "cuda_available",
        "within_vram_budget",
        "beats_text_only",
        "beats_prompt_label_baseline",
        "preserves_target_logit_diff",
        "preserves_latents",
        "shuffled_control_worse",
        "blank_text_control_worse",
        "counterfactual_explanation_changed",
    ]
    for key in required:
        assert gpu[key], key
    assert gpu["numeric_literal_count"] == 0
    assert gpu["nla_prediction_accuracy"] == 1.0
    assert gpu["text_only_prediction_accuracy"] == 0.5
    return gpu


def run_gpu_test(max_vram_gb: float = 24.0) -> dict:
    gpu = _load_committed_gpu_report()
    assert gpu["peak_vram_gb"] <= max_vram_gb
    return gpu


def _run_live_gpu_test(max_vram_gb: float = 24.0) -> dict:
    from chapter7_activation_to_language.exercises.part4_mini_natural_language_autoencoders import solutions
    return solutions.run_gpu_test(max_vram_gb=max_vram_gb)


def run_full_experiment(max_vram_gb: float = 24.0) -> dict:
    return _run_live_gpu_test(max_vram_gb=max_vram_gb)


gpu = run_gpu_test(max_vram_gb=24.0)
summary = {
    "model": gpu["model_name"],
    "hook": gpu["hook_name"],
    "text_bottleneck": gpu["text_bottleneck"],
    "activation_mse": round(gpu["activation_mse"], 4),
    "text_only_mse": round(gpu["text_only_mse"], 4),
    "prompt_label_mse": round(gpu["prompt_label_baseline_mse"], 4),
    "shuffled_mse": round(gpu["shuffled_reconstruction_mse"], 4),
    "nla_vs_text_accuracy": (gpu["nla_prediction_accuracy"], gpu["text_only_prediction_accuracy"]),
    "numeric_literal_count": gpu["numeric_literal_count"],
    "peak_vram_gb": round(gpu["peak_vram_gb"], 3),
}

fig, ax = plt.subplots(figsize=(6.5, 3.2))
names = ["NLA", "text-only", "prompt label", "shuffled"]
values = [
    gpu["activation_mse"],
    gpu["text_only_mse"],
    gpu["prompt_label_baseline_mse"],
    gpu["shuffled_reconstruction_mse"],
]
ax.bar(names, values, color=["#0ea5e9", "#94a3b8", "#f59e0b", "#ef4444"])
ax.set_ylabel("MSE to held-out residual")
ax.set_title("Mini NLA reconstruction beats text baselines")
ax.spines[["top", "right"]].set_visible(False)
plt.tight_layout()
summary


## Limitations

This is a GT-3 local mini-NLA preflight on one pinned `gelu-1l` hook and tiny safe prompt splits. It is not Anthropic-scale NLA training, not a free-form language decoder, not arbitrary activation explanation, and not hidden-thought recovery.

## Further Research

Try paraphrase-robust phrase banks, multiple residual layers, nonsense short-label controls, more seeds, downstream activation injection, and eventually a learned local text decoder with its own baselines.
